# Notebook 06 — EKF Core Ideas: Predict + Update

**Learning objectives**
- Define the EKF ingredients: state, covariance, process model, measurement model.
- Understand predict and update in equation form and plain language.
- Build a tiny 1D EKF for a constant state with noisy measurements.
- See how process noise `Q` and measurement noise `R` change responsiveness.

**Prerequisites**
- Notebooks 01–05 (Gaussian basics, covariance, Jacobians).


## Table of Contents

- [1) EKF ingredients in words](#1-ekf-ingredients-in-words)
- [2) Predict step (time update)](#2-predict-step-time-update)
- [3) Update step (measurement correction)](#3-update-step-measurement-correction)
- [4) Interactive exploration: effect of `Q` and `R`](#4-interactive-exploration-effect-of-q-and-r)
- [Exercises](#exercises)

---

## Navigation

- Previous: [ntbk-05-jacobians-for-robotics.ipynb](./ntbk-05-jacobians-for-robotics.ipynb)
- Next: [ntbk-07-ekf-2d-tracking-example.ipynb](./ntbk-07-ekf-2d-tracking-example.ipynb)

## Relevant source files (repo paths)
- `src/kiss_slam/ekf_slam.py`


In [ ]:
# Notebook setup (reproducible + matplotlib defaults)
import numpy as np
import matplotlib.pyplot as plt
from _notebook_utils import set_seed

SEED = 105
set_seed(SEED)
plt.rcdefaults()


## 1) EKF ingredients in words

We represent uncertainty with a Gaussian belief over state:

- State mean: $\hat{x}_k$ (our best estimate).
- State covariance: $P_k$ (how uncertain we are).

We also need two models:

1. **Process model**: $x_k = f(x_{k-1}, u_k) + w_k$.
   - This describes how state evolves after control/input $u_k$.
   - $w_k$ is process noise with covariance $Q_k$.

2. **Measurement model**: $z_k = h(x_k) + v_k$.
   - This maps state to what sensors should read.
   - $v_k$ is measurement noise with covariance $R_k$.


## 2) Predict step (time update)

EKF linearizes nonlinear models around the current estimate.

egin{aligned}
\hat{x}_{k|k-1} &= f(\hat{x}_{k-1|k-1}, u_k) \
F_k &= \left.rac{\partial f}{\partial x}ight|_{\hat{x}_{k-1|k-1},u_k} \
P_{k|k-1} &= F_k P_{k-1|k-1} F_k^T + Q_k
\end{aligned}

**In words:**
- Move the mean through the process model.
- Push covariance through the local linear map `F_k`.
- Add process noise `Q_k` because dynamics are never perfect.


## 3) Update step (measurement correction)

egin{aligned}
y_k &= z_k - h(\hat{x}_{k|k-1}) \quad 	ext{(innovation / residual)} \
H_k &= \left.rac{\partial h}{\partial x}ight|_{\hat{x}_{k|k-1}} \
S_k &= H_k P_{k|k-1} H_k^T + R_k \quad 	ext{(innovation covariance)} \
K_k &= P_{k|k-1} H_k^T S_k^{-1} \quad 	ext{(Kalman gain)} \
\hat{x}_{k|k} &= \hat{x}_{k|k-1} + K_k y_k \
P_{k|k} &= (I - K_k H_k)P_{k|k-1}
\end{aligned}

**In words:**
- Innovation says: *measurement minus prediction*.
- `S_k` is uncertainty in that innovation.
- `K_k` chooses how much to trust measurement versus prediction.
- Large `R_k` (noisy sensor) makes gain smaller; large `P` makes gain larger.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)


In [ ]:
class TinyEKF1D:
    """A minimal EKF for a constant 1D state x.

    Process model:      x_k = x_{k-1} + w_k
    Measurement model:  z_k = x_k + v_k

    This is linear, but we keep EKF structure to build intuition.
    """

    def __init__(self, x0: float, P0: float, Q: float, R: float):
        self.x = float(x0)
        self.P = float(P0)
        self.Q = float(Q)
        self.R = float(R)

    def predict(self):
        # f(x)=x, so predicted mean is unchanged.
        self.x = self.x
        # F=1 for constant-state model.
        self.P = self.P + self.Q

    def update(self, z: float):
        # h(x)=x, H=1
        y = z - self.x
        S = self.P + self.R
        K = self.P / S

        self.x = self.x + K * y
        self.P = (1.0 - K) * self.P
        return y, S, K


In [ ]:
# Simulate a constant truth and noisy measurements.
N = 80
x_true = 5.0
measurement_sigma = 0.8
z = x_true + np.random.randn(N) * measurement_sigma

ekf = TinyEKF1D(x0=0.0, P0=4.0, Q=0.02, R=measurement_sigma**2)

x_est, P_est, innovations, gains = [], [], [], []
for zk in z:
    ekf.predict()
    yk, Sk, Kk = ekf.update(zk)
    x_est.append(ekf.x)
    P_est.append(ekf.P)
    innovations.append(yk)
    gains.append(Kk)

x_est = np.asarray(x_est)
P_est = np.asarray(P_est)


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=True)

axes[0].plot(z, '.', alpha=0.45, label='measurement z')
axes[0].plot(x_est, '-', linewidth=2, label='EKF estimate')
axes[0].axhline(x_true, color='k', linestyle='--', label='true state')
axes[0].set_ylabel('state value')
axes[0].set_title('1D EKF: estimate converges despite noisy measurements')
axes[0].legend(loc='best')

axes[1].plot(P_est, color='tab:orange')
axes[1].set_ylabel('P (variance)')
axes[1].set_title('Covariance drops as evidence accumulates')

axes[2].plot(gains, color='tab:green')
axes[2].set_ylabel('Kalman gain K')
axes[2].set_xlabel('time step')
axes[2].set_title('Gain decreases as filter becomes confident')

plt.tight_layout()


## 4) Interactive exploration: effect of `Q` and `R`

Interpretation:
- Bigger `Q`: trust dynamics less, so uncertainty grows faster and filter reacts more.
- Bigger `R`: trust measurements less, so updates are weaker and smoother.


In [ ]:
def run_filter(Q, R, seed=0):
    np.random.seed(seed)
    z = x_true + np.random.randn(N) * measurement_sigma
    f = TinyEKF1D(x0=0.0, P0=4.0, Q=Q, R=R)
    est = []
    for zk in z:
        f.predict()
        f.update(zk)
        est.append(f.x)
    return z, np.asarray(est)

settings = [
    (0.001, 0.64, 'low Q, nominal R'),
    (0.10, 0.64, 'high Q, nominal R'),
    (0.02, 0.10, 'nominal Q, low R'),
    (0.02, 2.00, 'nominal Q, high R'),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
for ax, (Q, R, name) in zip(axes.ravel(), settings):
    z_plot, est = run_filter(Q, R, seed=7)
    ax.plot(z_plot, '.', alpha=0.25, label='z')
    ax.plot(est, '-', linewidth=2, label='estimate')
    ax.axhline(x_true, color='k', linestyle='--', linewidth=1)
    ax.set_title(f'{name}\nQ={Q}, R={R}')

axes[0,0].legend(loc='lower right')
plt.suptitle('Responsiveness vs smoothness from Q/R tuning')
plt.tight_layout()


## Exercises

1. Set `Q=0` and very small `R`. What happens to gain and convergence speed?
2. Set `R` much larger than true sensor variance. Does the estimate lag?
3. Start with a huge `P0` vs tiny `P0`. How does early behavior differ?

<details>
<summary><b>Optional solution ideas</b></summary>

- `Q=0` can make the filter overconfident over long horizons if model errors exist.
- Too-large `R` makes correction weak, so estimate is smoother but slower.
- Large `P0` gives larger early gain, so fast adaptation from poor initialization.

</details>


## Exercise Solutions

<details>
<summary>Show solution sketches</summary>

- Re-run the exercise code cells and compare your intermediate values to the reference outputs printed in this notebook.
- Check shapes (`mean`, `covariance`, Jacobians) first; most EKF mistakes are shape/sign issues.
- Use the same `SEED` from the setup cell to keep your run deterministic while debugging.

</details>
